# SQLAlchemy relationships in depth, and Alembic migrations
* lesson 40 got us through the fundamentals: the `Engine`, Core's expression-based `Table`/`insert()`/`select()`, and the modern ORM's `Mapped`/`mapped_column()` style with a simple one-to-many `relationship()` and full CRUD through a `Session`
* this lesson goes deeper on relationships specifically - many-to-many, the difference between lazy and eager loading (and the infamous "N+1 query problem"), cascading deletes, and doing joins/aggregates through the ORM - and then introduces **Alembic**, the tool that manages a real, EVOLVING database schema over time

## many-to-many relationships: an association table
* recap lesson 40: a one-to-many relationship (one `Author` has many `Book`s) only needed ONE foreign key column, sitting on the "many" side (`books.author_id`)
* many-to-many needs a THIRD table in between, because neither side alone has anywhere to put "the" foreign key - a `Book` can have several `Tag`s, and a `Tag` can apply to several `Book`s, so we need a table that just records which book-id goes with which tag-id
* that in-between table (often called an "association" or "junction" table) usually has no interesting data of its own - it's ONLY the pairing - so we describe it as a plain `Table` object (recap lesson 40's Core section), never as its own mapped class
* we hand that `Table` to `relationship(secondary=...)` on BOTH sides, and `back_populates` keeps both directions in sync automatically, exactly like the one-to-many case

In [1]:
from sqlalchemy import create_engine, String, Integer, ForeignKey, Table, Column
from sqlalchemy.orm import declarative_base, Mapped, mapped_column, relationship

# fresh in-memory engine for this whole lesson - same reasoning as lesson 40: fully self-contained, offline, nothing to clean up here
engine = create_engine("sqlite:///:memory:")
Base = declarative_base()

# the association ("junction"/"link") table for the many-to-many relationship between books and tags.
# this is a plain Table (recap lesson 40's Core section), NOT a mapped class like Author/Book/Tag below - it carries
# no columns worth turning into python attributes, it ONLY records "this book_id pairs with that tag_id"
book_tags = Table(
    "book_tags",
    Base.metadata,
    Column("book_id", ForeignKey("books.id"), primary_key=True),  # both columns TOGETHER form the primary key here -
    Column("tag_id", ForeignKey("tags.id"), primary_key=True),    # that combo is exactly what stops the same pairing being inserted twice
)


class Author(Base):
    __tablename__ = "authors"
    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(100))
    # plain one-to-many, no cascade configured (yet!) - we come back to this EXACT relationship in the cascades
    # section further down, once we have seen what deleting an author does to their books WITHOUT one
    books: Mapped[list["Book"]] = relationship(back_populates="author")


class Book(Base):
    __tablename__ = "books"
    id: Mapped[int] = mapped_column(primary_key=True)
    title: Mapped[str] = mapped_column(String(200))
    author_id: Mapped[int] = mapped_column(ForeignKey("authors.id"))
    author: Mapped["Author"] = relationship(back_populates="books")

    # THIS is the actual many-to-many relationship - secondary= points at the association TABLE (not a mapped class),
    # that is the one visual detail that tells you "this relationship is many-to-many" instead of one-to-many
    tags: Mapped[list["Tag"]] = relationship(secondary=book_tags, back_populates="books")


class Tag(Base):
    __tablename__ = "tags"
    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(50), unique=True)

    # the OTHER side of the SAME many-to-many, same secondary= table - SQLAlchemy figures out which of book_tags' two
    # foreign keys belongs to which side on its own here, since there are only 2 possible foreign keys to match up
    books: Mapped[list["Book"]] = relationship(secondary=book_tags, back_populates="tags")


Base.metadata.create_all(engine)
print("tables created:", list(Base.metadata.tables.keys()))

tables created: ['book_tags', 'authors', 'books', 'tags']


In [2]:
from sqlalchemy.orm import Session

with Session(engine) as session:
    # one author with two books - same setup shape as lesson 40, nothing new on this side yet
    le_guin = Author(name="Ursula K. Le Guin")
    left_hand = Book(title="The Left Hand of Darkness", author=le_guin)
    dispossessed = Book(title="The Dispossessed", author=le_guin)

    # a handful of Tag objects - the whole point of many-to-many is that these get SHARED/reused across several books
    scifi = Tag(name="sci-fi")
    classic = Tag(name="classic")
    political = Tag(name="political")

    # .append()-ing straight onto the .tags list is all it takes - SQLAlchemy inserts the matching book_tags rows for
    # us behind the scenes, we never touch that association table by hand, exactly like appending to a one-to-many list
    left_hand.tags.append(scifi)
    left_hand.tags.append(political)
    dispossessed.tags.append(scifi)
    dispossessed.tags.append(classic)
    dispossessed.tags.append(political)

    session.add_all([le_guin, left_hand, dispossessed])
    session.commit()
    left_hand_id, dispossessed_id = left_hand.id, dispossessed.id

# a couple MORE authors, each with several books, and no tags - we are setting this data up now for the eager/lazy
# loading demo further down, where having more than one author with multiple books each actually matters
with Session(engine) as session:
    for name, titles in [
        ("Isaac Asimov", ["Foundation", "I, Robot", "Nightfall"]),
        ("Frank Herbert", ["Dune", "Dune Messiah"]),
    ]:
        author = Author(name=name)
        for title in titles:
            author.books.append(Book(title=title))
        session.add(author)
    session.commit()

print("data inserted: 3 authors total, 7 books total, 3 tags, several book<->tag pairings")

data inserted: 3 authors total, 7 books total, 3 tags, several book<->tag pairings


In [3]:
from sqlalchemy import select

with Session(engine) as session:
    # query "every book tagged sci-fi" - Book.tags.any(...) is the many-to-many equivalent of a SQL "WHERE EXISTS"
    # subquery, it walks through the book_tags association table for us without us ever writing that join ourselves
    scifi_books = session.scalars(select(Book).where(Book.tags.any(Tag.name == "sci-fi"))).all()
    print("books tagged 'sci-fi':", [book.title for book in scifi_books])

    # the exact same relationship works from the OTHER direction too, straight off a Tag object we already have
    political_tag = session.scalars(select(Tag).where(Tag.name == "political")).one()
    print(f"books tagged {political_tag.name!r}:", [book.title for book in political_tag.books])

    # one single book's full list of tags, from yet another angle - same relationship, just reading the other attribute
    dispossessed_book = session.get(Book, dispossessed_id)
    print(f"{dispossessed_book.title!r} is tagged:", [tag.name for tag in dispossessed_book.tags])

books tagged 'sci-fi': ['The Left Hand of Darkness', 'The Dispossessed']
books tagged 'political': ['The Dispossessed', 'The Left Hand of Darkness']
'The Dispossessed' is tagged: ['sci-fi', 'political', 'classic']


## lazy vs eager loading: the N+1 query problem
* by default, every `relationship()` we have used so far (in lesson 40 and above) is **lazy** - it does NOT get loaded together with its parent object, it fires its OWN, separate `SELECT` only the FIRST time you actually access it, e.g. `author.books`
* that is usually completely fine for a single object - but it becomes a real performance trap inside a loop: load a LIST of N authors (1 query), then access `.books` on EACH one inside a loop -> N MORE separate queries fire, one per author -> **N+1 queries total** for something that could have been done in 1 or 2
* this is called the **N+1 query problem**, and it is one of THE most common real-world ORM performance bugs, in essentially every language/ORM that offers lazy relationships
* the fix is **eager loading**: tell the ORM upfront, as part of the SAME query, "also fetch this relationship right now" via `.options(...)` on your `select()` - so it never needs a separate round-trip per object

In [4]:
from sqlalchemy import event

# a small, reliable way to COUNT exactly how many SQL statements actually get sent to sqlite - much more trustworthy
# than just eyeballing echo=True output by hand. "before_cursor_execute" is an event SQLAlchemy fires right before
# EVERY single statement it sends down to the underlying database driver, no matter which layer triggered it
executed_statements = []


def _record_statement(conn, cursor, statement, parameters, context, executemany):
    executed_statements.append(statement)


event.listen(engine, "before_cursor_execute", _record_statement)

engine.echo = True  # ALSO turn on SQL logging just for this section, so we can literally watch every query print below

executed_statements.clear()  # reset the counter right before the exact scenario we actually want to measure
with Session(engine) as session:
    all_authors = session.scalars(select(Author)).all()  # query #1 (visible above): just the 3 authors themselves
    for author in all_authors:
        # accessing .books HERE, inside the loop, is the trap - each access triggers its OWN separate SELECT,
        # because this relationship is lazy and nothing has told it to load anything upfront yet
        print(author.name, "wrote", len(author.books), "books")

engine.echo = False  # turn logging back off again, so the rest of the lesson does not stay noisy
print()
print(f"total SQL statements executed for {len(all_authors)} authors:", len(executed_statements))
print("that is 1 query for the authors, PLUS 1 SEPARATE query per author for their books - the classic N+1 pattern")

2026-09-26 11:57:26,642 INFO sqlalchemy.engine.Engine BEGIN (implicit)


2026-09-26 11:57:26,643 INFO sqlalchemy.engine.Engine SELECT authors.id, authors.name 
FROM authors


2026-09-26 11:57:26,644 INFO sqlalchemy.engine.Engine [generated in 0.00089s] ()


2026-09-26 11:57:26,646 INFO sqlalchemy.engine.Engine SELECT books.id, books.title, books.author_id 
FROM books 
WHERE ? = books.author_id


2026-09-26 11:57:26,647 INFO sqlalchemy.engine.Engine [generated in 0.00081s] (1,)


...

2026-09-26 11:57:26,651 INFO sqlalchemy.engine.Engine [cached since 0.004599s ago] (2,)


Isaac Asimov wrote 3 books
2026-09-26 11:57:26,653 INFO sqlalchemy.engine.Engine SELECT books.id, books.title, books.author_id 
FROM books 
WHERE ? = books.author_id


2026-09-26 11:57:26,654 INFO sqlalchemy.engine.Engine [cached since 0.007141s ago] (3,)


Frank Herbert wrote 2 books
2026-09-26 11:57:26,655 INFO sqlalchemy.engine.Engine ROLLBACK



total SQL statements executed for 3 authors: 4
that is 1 query for the authors, PLUS 1 SEPARATE query per author for their books - the classic N+1 pattern


In [5]:
from sqlalchemy.orm import selectinload

executed_statements.clear()
with Session(engine) as session:
    # .options(selectinload(Author.books)) tells the ORM upfront: "when loading Authors, ALSO fetch their books" -
    # but do it as ONE extra bulk query (SELECT ... FROM books WHERE author_id IN (...all the loaded author ids...)),
    # not one query per author. this is almost always the right default fix for a one-to-many/many-to-many N+1 problem
    query = select(Author).options(selectinload(Author.books))
    all_authors = session.scalars(query).all()
    for author in all_authors:
        print(author.name, "wrote", len(author.books), "books")  # no NEW query fires here anymore - already loaded above

print()
print("total SQL statements with selectinload():", len(executed_statements))
print("down from N+1 to just 2 total - 1 for the authors, 1 bulk query that grabs ALL of their books together")

Ursula K. Le Guin wrote 2 books
Isaac Asimov wrote 3 books
Frank Herbert wrote 2 books

total SQL statements with selectinload(): 2
down from N+1 to just 2 total - 1 for the authors, 1 bulk query that grabs ALL of their books together


In [6]:
from sqlalchemy.orm import joinedload

executed_statements.clear()
with Session(engine) as session:
    # joinedload takes a DIFFERENT approach again: ONE single query total, using a LEFT OUTER JOIN to pull authors
    # and books together in one result set - great when there is a small number of "many" rows per parent, but the
    # trade-off is that the result set repeats every author's columns once per matching book row
    query = select(Author).options(joinedload(Author.books))
    # .unique() is REQUIRED here (SQLAlchemy raises an error and tells you so if you forget it) - it collapses the
    # duplicate author rows the JOIN just created back down to one Author object each, before turning it into a list
    all_authors = session.scalars(query).unique().all()
    for author in all_authors:
        print(author.name, "wrote", len(author.books), "books")

print()
print("total SQL statements with joinedload():", len(executed_statements))
print("just 1 single query total - fewer database round-trips, at the cost of that wider/duplicated result set")

Ursula K. Le Guin wrote 2 books
Isaac Asimov wrote 3 books
Frank Herbert wrote 2 books

total SQL statements with joinedload(): 1
just 1 single query total - fewer database round-trips, at the cost of that wider/duplicated result set


## cascades: what happens to children when a parent gets deleted
* recap: `Book.author_id` is a foreign key we declared as `Mapped[int]` (not `Optional[int]`) - it is `NOT NULL`, it MUST always point at a real author
* so what SHOULD happen if we delete an `Author` who still has `Book`s? there is no single universally correct answer - it depends entirely on the application - but SQLAlchemy has to do SOMETHING, and the default behaviour is worth understanding before you rely on it
* `relationship()`'s `cascade=` parameter is how WE tell the ORM what "delete this parent" should mean for its children - `"all, delete-orphan"` is the setting that says "when I delete the parent (or remove a child from its list), delete that child row too"
* lets see the default first, on the exact `Author`/`Book` relationship from the start of this lesson (which has no cascade configured at all), then compare it to the same setup WITH `cascade="all, delete-orphan"` turned on

In [7]:
from sqlalchemy.exc import IntegrityError

with Session(engine) as session:
    le_guin_author = session.scalars(select(Author).where(Author.name == "Ursula K. Le Guin")).one()
    le_guin_id = le_guin_author.id  # grab this now, before the session below closes and the object detaches
    print("before delete, her books:", [book.title for book in le_guin_author.books])

    session.delete(le_guin_author)  # tell the session we want this Author gone
    try:
        # without any cascade configured, SQLAlchemy's DEFAULT move for the now-orphaned books is to try setting
        # their author_id to NULL (effectively "un-link" them, rather than delete them outright) - but we declared
        # author_id as NOT NULL back when we defined Book, so that UPDATE itself fails
        session.commit()
        print("commit succeeded")
    except IntegrityError as error:
        session.rollback()  # undo the WHOLE failed transaction - neither the author nor the books actually changed
        print("delete FAILED with an IntegrityError, exactly because of that NOT NULL foreign key:")
        print(error)

with Session(engine) as session:
    # confirm NOTHING actually changed - the rollback really did undo everything, author and books are both still there
    print("author still exists:", session.get(Author, le_guin_id) is not None)
    still_there = session.scalars(select(Book).where(Book.author_id == le_guin_id)).all()
    print("her books still exist:", [book.title for book in still_there])

before delete, her books: ['The Left Hand of Darkness', 'The Dispossessed']
delete FAILED with an IntegrityError, exactly because of that NOT NULL foreign key:
(sqlite3.IntegrityError) NOT NULL constraint failed: books.author_id
[SQL: UPDATE books SET author_id=? WHERE books.id = ?]
[parameters: [(None, 1), (None, 2)]]
(Background on this error at: https://sqlalche.me/e/21/gkpj)
author still exists: True
her books still exist: ['The Left Hand of Darkness', 'The Dispossessed']


In [8]:
class CascadingAuthor(Base):
    __tablename__ = "cascading_authors"  # fresh table names so this does not collide with the plain Author/Book above
    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(100))
    # THE only difference from the plain Author class above: cascade="all, delete-orphan" - "all" cascades every normal
    # session operation (save, update, merge, ...) the same way the default already did, and "delete-orphan"
    # ADDITIONALLY says "if a Book gets removed from this list, or its parent Author gets deleted, delete that Book row too"
    books: Mapped[list["CascadingBook"]] = relationship(back_populates="author", cascade="all, delete-orphan")


class CascadingBook(Base):
    __tablename__ = "cascading_books"
    id: Mapped[int] = mapped_column(primary_key=True)
    title: Mapped[str] = mapped_column(String(200))
    author_id: Mapped[int] = mapped_column(ForeignKey("cascading_authors.id"))
    author: Mapped["CascadingAuthor"] = relationship(back_populates="books")


Base.metadata.create_all(engine)

with Session(engine) as session:
    cascade_author = CascadingAuthor(name="Cascade Demo Author")
    cascade_author.books.append(CascadingBook(title="Book A"))
    cascade_author.books.append(CascadingBook(title="Book B"))
    session.add(cascade_author)
    session.commit()
    cascade_author_id = cascade_author.id

with Session(engine) as session:
    author_to_delete = session.get(CascadingAuthor, cascade_author_id)
    session.delete(author_to_delete)
    session.commit()  # this time NO IntegrityError - cascade="all, delete-orphan" deletes the books FIRST, then the author
    print("delete succeeded, no error this time")

with Session(engine) as session:
    remaining = session.scalars(select(CascadingBook)).all()
    print("CascadingBook rows left over after deleting their author:", remaining)  # empty list - they really are gone

delete succeeded, no error this time
CascadingBook rows left over after deleting their author: []


## joins and aggregates through the ORM
* recap lesson 13's raw SQL `JOIN`, and this lesson's own relationship-based access (`author.books`) - but sometimes you want to filter or aggregate ACROSS tables directly inside the query itself, instead of walking python objects one by one after the fact
* `select(Model).join(OtherModel)` does exactly what it looks like - an explicit SQL `JOIN`, expressed as a chain of ORM query-builder calls instead of a raw `JOIN ... ON ...` string
* `func.count()`, `.group_by()` and `.order_by()` (recap lesson 13's `GROUP BY`) all work the same way here as they did in raw SQL, just written as python expressions like the rest of Core/the ORM

In [9]:
from sqlalchemy import func

with Session(engine) as session:
    # select(Author).join(Book) - an explicit INNER JOIN, same idea as lesson 13's "JOIN books ON books.author_id = authors.id",
    # just expressed as a python method chain instead of a raw JOIN...ON string. .distinct() matters here because a
    # JOIN produces one row PER matching book, and "Dune"/"Dune Messiah" both belong to the same author (Frank Herbert)
    query = select(Author).join(Book).where(Book.title.like("Dune%")).distinct()
    dune_authors = session.scalars(query).all()
    print("authors with a book titled like 'Dune%':", [a.name for a in dune_authors])

    # selecting EXPRESSIONS (not whole model objects this time) - Author.name together with an AGGREGATE,
    # func.count(Book.id) counts how many book rows joined to each author, .group_by() collapses rows per author,
    # exactly like SQL GROUP BY (recap lesson 13) - .order_by() sorts by that same count, most prolific author first
    counts_query = (
        select(Author.name, func.count(Book.id).label("book_count"))
        .join(Book)
        .group_by(Author.name)
        .order_by(func.count(Book.id).desc())
    )
    for name, book_count in session.execute(counts_query):
        print(f"{name}: {book_count} books")

authors with a book titled like 'Dune%': ['Frank Herbert']
Isaac Asimov: 3 books
Ursula K. Le Guin: 2 books
Frank Herbert: 2 books


## Alembic migrations: version-controlled schema changes
* recap: every `create_all()` we have called so far (lesson 40 and above) only ever CREATES tables that do not exist yet - it does absolutely nothing to a table that already exists, even if we have since changed our mapped class to add, rename or remove a column
* in any real, evolving application that is a real problem: you deploy a new version of your code with a changed model, but the ACTUAL database out there in production still has the OLD schema, and it usually already holds real data you cannot just throw away by dropping and recreating tables
* **Alembic** is SQLAlchemy's official migration tool - a **migration** is simply a small python script with an `upgrade()` function (apply this one change) and a matching `downgrade()` function (undo it again). Alembic keeps a special `alembic_version` table INSIDE the actual database, recording exactly which migration it is currently at, so it always knows what to run next
* a real project normally keeps its OWN dedicated folder with `alembic.ini`/`env.py`/`versions/` living permanently on disk, and you would run `alembic revision`/`alembic upgrade head` from a terminal. for THIS lesson we build that exact same folder structure ourselves, by hand, inside a throwaway temporary directory - purely so the whole example stays runnable end-to-end inside this notebook - and we delete it again at the very end

In [10]:
import tempfile, os, shutil
from alembic.config import Config
from alembic import command
from sqlalchemy import create_engine, inspect

# a real project keeps this folder permanently in its repo - we build the same 3 files (alembic.ini, env.py,
# script.py.mako) by hand in a throwaway temp directory instead, purely so this lesson stays self-contained
workdir = tempfile.mkdtemp(prefix="alembic_demo_")
db_path = os.path.join(workdir, "demo.db")
db_url = f"sqlite:///{db_path}"  # a REAL file this time, not :memory: - alembic needs a file it can reconnect to across separate command.* calls

versions_dir = os.path.join(workdir, "versions")  # every individual migration script lives in here, one file per revision
os.makedirs(versions_dir)

# alembic.ini is alembic's own top-level config file - script_location tells it where to find env.py/versions/ etc
alembic_ini = os.path.join(workdir, "alembic.ini")
with open(alembic_ini, "w") as f:
    f.write(f"[alembic]\nscript_location = {workdir}\n")

# env.py is the script alembic ACTUALLY runs every single time it upgrades/downgrades - it connects to the real
# database and then hands control over to alembic's migration runner. target_metadata=None here means "no
# autogenerate diffing" - we are writing every upgrade()/downgrade() body ourselves by hand below, for full clarity
with open(os.path.join(workdir, "env.py"), "w") as f:
    f.write("""
from alembic import context
from sqlalchemy import create_engine

config = context.config

def run_migrations_online():
    connectable = create_engine(config.get_main_option("sqlalchemy.url"))
    with connectable.connect() as connection:
        context.configure(connection=connection, target_metadata=None)
        with context.begin_transaction():
            context.run_migrations()

run_migrations_online()
""")

# script.py.mako is the TEMPLATE alembic fills in every single time we generate a new migration file with command.revision()
with open(os.path.join(workdir, "script.py.mako"), "w") as f:
    f.write("""\"\"\"${message}\"\"\"
revision = ${repr(up_revision)}
down_revision = ${repr(down_revision)}
from alembic import op
import sqlalchemy as sa
${imports if imports else ""}

def upgrade():
    ${upgrades if upgrades else "pass"}

def downgrade():
    ${downgrades if downgrades else "pass"}
""")

cfg = Config(alembic_ini)                        # load the ini file we just wrote
cfg.set_main_option("sqlalchemy.url", db_url)     # tell alembic which database to actually migrate
cfg.set_main_option("script_location", workdir)   # and confirm again where its scripts live

# command.revision() generates a brand new, EMPTY migration file and hands us back its path - normally you would
# open that file in your own editor next; here we just overwrite it straight away with our own upgrade()/downgrade()
first_revision = command.revision(cfg, message="create employees table", head="base")
with open(first_revision.path, "w") as f:
    f.write(f"""\"\"\"create employees table\"\"\"
revision = {first_revision.revision!r}
down_revision = None
from alembic import op
import sqlalchemy as sa

def upgrade():
    # op.create_table() is alembic's own schema-change API - notice this reads almost exactly like Core's Table() from lesson 40
    op.create_table(
        "employees",
        sa.Column("id", sa.Integer, primary_key=True),
        sa.Column("name", sa.String(50)),
    )

def downgrade():
    op.drop_table("employees")  # undoing "create a table" is simply "drop that table" again
""")

command.upgrade(cfg, "head")  # actually RUN upgrade() against our real sqlite file - "head" means "the newest migration we have"
engine_alembic = create_engine(db_url)
print("tables after migration 1:", inspect(engine_alembic).get_table_names())
print("employees columns after migration 1:", [c["name"] for c in inspect(engine_alembic).get_columns("employees")])

Generating /tmp/alembic_demo_iohlew0y/versions/a544a07a4c7a_create_employees_table.py ...

  done


tables after migration 1: ['alembic_version', 'employees']
employees columns after migration 1: ['id', 'name']


In [11]:
# a SECOND migration - this is the whole point of alembic: schema changes are a numbered, ORDERED sequence, each one
# building on top of the last, and each one knows exactly which migration came directly before it via down_revision
second_revision = command.revision(cfg, message="add email column", head="head")
with open(second_revision.path, "w") as f:
    f.write(f"""\"\"\"add email column\"\"\"
revision = {second_revision.revision!r}
down_revision = {first_revision.revision!r}
from alembic import op
import sqlalchemy as sa

def upgrade():
    # op.add_column() - THIS is the important difference from create_all(): create_all() would do NOTHING here,
    # because the employees table already exists. a migration can ALTER a table that is already there, without
    # ever losing the data already stored in it
    op.add_column("employees", sa.Column("email", sa.String(120)))

def downgrade():
    op.drop_column("employees", "email")  # undo "add a column" by dropping that same column again
""")

command.upgrade(cfg, "head")  # runs ONLY the new migration - alembic already knows (via its alembic_version table) that migration 1 is done
print("employees columns after migration 2:", [c["name"] for c in inspect(engine_alembic).get_columns("employees")])

# to prove no data got lost along the way, lets actually insert a row now that both columns exist, and read it back
from sqlalchemy import text

with engine_alembic.connect() as connection:
    connection.execute(
        text("INSERT INTO employees (name, email) VALUES (:name, :email)"),
        {"name": "Grace Hopper", "email": "grace@example.com"},
    )
    connection.commit()
    surviving_row = connection.execute(text("SELECT name, email FROM employees")).one()
    print("row surviving across both migrations:", surviving_row)

Generating /tmp/alembic_demo_iohlew0y/versions/c3558ded517a_add_email_column.py ...

  done
employees columns after migration 2: ['id', 'name', 'email']
row surviving across both migrations: ('Grace Hopper', 'grace@example.com')


In [12]:
command.downgrade(cfg, "base")  # "base" means "undo everything, all the way back to before migration 1 ever ran"
remaining_tables = inspect(engine_alembic).get_table_names()
print("tables after downgrading all the way to base:", remaining_tables)
print("employees is gone - alembic_version stays behind, that one is alembic's OWN bookkeeping table, not ours")

# always clean up any temp files/directories a demo like this creates - never leave throwaway files behind in the repo
shutil.rmtree(workdir)
print("temp alembic project directory removed:", not os.path.exists(workdir))

tables after downgrading all the way to base: ['alembic_version']
employees is gone - alembic_version stays behind, that one is alembic's OWN bookkeeping table, not ours
temp alembic project directory removed: True


## quick comparison to other languages
* alembic-style migrations are not a SQLAlchemy invention - essentially every serious backend ecosystem has converged on the same idea: **Django** (python's other major web framework) ships its OWN built-in migrations system, **Ruby on Rails**' ActiveRecord migrations are arguably THE most famous/influential example of this whole pattern, C#'s **Entity Framework** has "EF Migrations", in Node.js **Sequelize**/**Prisma**/**TypeORM**/**Knex** each bring their own migration tooling, Go has `golang-migrate`, and Rust has `sqlx-cli`/**Diesel** - all of them doing the exact same job for their own ecosystem
* the underlying idea is close to universal across ALL of these tools: a numbered, ORDERED sequence of up/down schema-change scripts, tracked in a special table (our `alembic_version`) so the tool always knows exactly what state the database is currently in, and exactly what to run next
* many-to-many association tables and the N+1 query problem are not python-specific either - **Hibernate** (java's SQLAlchemy-equivalent, recap lesson 40's comparison section) famously has the EXACT same N+1 problem, and the EXACT same fix: an eager "fetch join", which is basically `joinedload()` under a different name

## Exercises
1. add a THIRD `Tag` to one of the books from earlier in this lesson (e.g. `left_hand_id`), and confirm that querying books by that new tag now correctly includes it
2. deliberately trigger and OBSERVE an N+1 query pattern on a fresh small example, with `echo=True` on the engine and a query counter like we built above, then fix it with `selectinload()` and show the query count improve
3. design your OWN fresh one-to-many relationship (any two related things you like), add `cascade="all, delete-orphan"` to it, and demonstrate that deleting the parent object also removes its children automatically
4. write one MORE Alembic migration that renames a column (recap `op.batch_alter_table(...)` - needed because SQLite has no native `ALTER COLUMN`), apply it, and confirm the renamed column exists via `inspect(engine).get_columns(...)`

In [13]:
# TODO: solve exercise 1 here (add a third Tag to a book, confirm querying by that tag includes it)


# TODO: solve exercise 2 here (trigger + observe N+1 with echo=True, then fix it with selectinload and compare query counts)


# TODO: solve exercise 3 here (your own one-to-many relationship with cascade="all, delete-orphan", deleting the parent removes the children)


# TODO: solve exercise 4 here (one more Alembic migration that renames a column, then confirm via inspect(engine).get_columns(...))

### sample solutions
* try the exercises above yourself first, only look here if you are stuck or want to compare your approach

In [14]:
# sample solution 1 - add a third tag to "The Left Hand of Darkness" (it already has sci-fi + political from earlier)
with Session(engine) as session:
    book_for_new_tag = session.get(Book, left_hand_id)
    utopian_tag = Tag(name="utopian")
    book_for_new_tag.tags.append(utopian_tag)  # same append-onto-the-relationship-list trick as earlier in the lesson
    session.commit()

with Session(engine) as session:
    utopian_books = session.scalars(select(Book).where(Book.tags.any(Tag.name == "utopian"))).all()
    print("books tagged 'utopian':", [b.title for b in utopian_books])


# sample solution 2 - trigger + observe N+1 on the existing authors/books, then fix with selectinload
exercise_statements = []


def _record_exercise_statement(conn, cursor, statement, parameters, context, executemany):
    exercise_statements.append(statement)


event.listen(engine, "before_cursor_execute", _record_exercise_statement)

engine.echo = True
exercise_statements.clear()
with Session(engine) as session:
    for author in session.scalars(select(Author)).all():
        _ = author.books  # deliberately trigger the lazy load, one separate query per author
engine.echo = False
print()
print("exercise 2 - N+1 query count:", len(exercise_statements))

exercise_statements.clear()
with Session(engine) as session:
    for author in session.scalars(select(Author).options(selectinload(Author.books))).all():
        _ = author.books  # already loaded together with the authors, no extra query fires here
print("exercise 2 - selectinload query count:", len(exercise_statements))

event.remove(engine, "before_cursor_execute", _record_exercise_statement)  # detach this listener again, tidy up after ourselves


# sample solution 3 - a fresh Playlist -> Track one-to-many relationship, with cascade="all, delete-orphan"
class Playlist(Base):
    __tablename__ = "playlists"
    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(100))
    tracks: Mapped[list["Track"]] = relationship(back_populates="playlist", cascade="all, delete-orphan")


class Track(Base):
    __tablename__ = "tracks"
    id: Mapped[int] = mapped_column(primary_key=True)
    title: Mapped[str] = mapped_column(String(200))
    playlist_id: Mapped[int] = mapped_column(ForeignKey("playlists.id"))
    playlist: Mapped["Playlist"] = relationship(back_populates="tracks")


Base.metadata.create_all(engine)

with Session(engine) as session:
    playlist = Playlist(name="Focus Mix")
    playlist.tracks.append(Track(title="Track One"))
    playlist.tracks.append(Track(title="Track Two"))
    session.add(playlist)
    session.commit()
    playlist_id = playlist.id

with Session(engine) as session:
    session.delete(session.get(Playlist, playlist_id))
    session.commit()  # cascade="all, delete-orphan" deletes both tracks automatically, no IntegrityError here

with Session(engine) as session:
    print("exercise 3 - tracks left after deleting their playlist:", session.scalars(select(Track)).all())


# sample solution 4 - one more alembic migration, in a fresh little project, this time RENAMING a column
workdir2 = tempfile.mkdtemp(prefix="alembic_exercise_")
db_url2 = f"sqlite:///{os.path.join(workdir2, 'exercise.db')}"
os.makedirs(os.path.join(workdir2, "versions"))

alembic_ini2 = os.path.join(workdir2, "alembic.ini")
with open(alembic_ini2, "w") as f:
    f.write(f"[alembic]\nscript_location = {workdir2}\n")

# render_as_batch=True matters specifically for SQLite - it has no native ALTER COLUMN, so alembic instead rebuilds
# the whole table under the hood (copy data -> drop old table -> create new table -> copy data back) whenever we
# use batch_alter_table() below, which is exactly what makes a column rename possible on SQLite at all
with open(os.path.join(workdir2, "env.py"), "w") as f:
    f.write("""
from alembic import context
from sqlalchemy import create_engine

config = context.config

def run_migrations_online():
    connectable = create_engine(config.get_main_option("sqlalchemy.url"))
    with connectable.connect() as connection:
        context.configure(connection=connection, target_metadata=None, render_as_batch=True)
        with context.begin_transaction():
            context.run_migrations()

run_migrations_online()
""")

with open(os.path.join(workdir2, "script.py.mako"), "w") as f:
    f.write("""\"\"\"${message}\"\"\"
revision = ${repr(up_revision)}
down_revision = ${repr(down_revision)}
from alembic import op
import sqlalchemy as sa
${imports if imports else ""}

def upgrade():
    ${upgrades if upgrades else "pass"}

def downgrade():
    ${downgrades if downgrades else "pass"}
""")

cfg2 = Config(alembic_ini2)
cfg2.set_main_option("sqlalchemy.url", db_url2)
cfg2.set_main_option("script_location", workdir2)

setup_revision = command.revision(cfg2, message="create customers table", head="base")
with open(setup_revision.path, "w") as f:
    f.write(f"""\"\"\"create customers table\"\"\"
revision = {setup_revision.revision!r}
down_revision = None
from alembic import op
import sqlalchemy as sa

def upgrade():
    op.create_table("customers", sa.Column("id", sa.Integer, primary_key=True), sa.Column("name", sa.String(100)))

def downgrade():
    op.drop_table("customers")
""")
command.upgrade(cfg2, "head")

rename_revision = command.revision(cfg2, message="rename name to full_name", head="head")
with open(rename_revision.path, "w") as f:
    f.write(f"""\"\"\"rename name to full_name\"\"\"
revision = {rename_revision.revision!r}
down_revision = {setup_revision.revision!r}
from alembic import op
import sqlalchemy as sa

def upgrade():
    # batch_alter_table opens a "batch" context - inside it, alter_column can rename a column even on SQLite,
    # which normally offers no ALTER COLUMN support at all (recap the render_as_batch=True comment in env.py above)
    with op.batch_alter_table("customers") as batch_op:
        batch_op.alter_column("name", new_column_name="full_name")

def downgrade():
    with op.batch_alter_table("customers") as batch_op:
        batch_op.alter_column("full_name", new_column_name="name")
""")
command.upgrade(cfg2, "head")

engine2 = create_engine(db_url2)
print("exercise 4 - customers columns after the rename migration:", [c["name"] for c in inspect(engine2).get_columns("customers")])

shutil.rmtree(workdir2)  # clean up this exercise's temp project too, exactly like the main lesson did above
print("exercise 4 - temp directory removed:", not os.path.exists(workdir2))

books tagged 'utopian': ['The Left Hand of Darkness']
2026-09-26 11:57:27,002 INFO sqlalchemy.engine.Engine BEGIN (implicit)


2026-09-26 11:57:27,003 INFO sqlalchemy.engine.Engine SELECT authors.id, authors.name 
FROM authors


2026-09-26 11:57:27,004 INFO sqlalchemy.engine.Engine [cached since 0.3602s ago] ()


2026-09-26 11:57:27,005 INFO sqlalchemy.engine.Engine SELECT books.id, books.title, books.author_id 
FROM books 
WHERE ? = books.author_id


2026-09-26 11:57:27,005 INFO sqlalchemy.engine.Engine [cached since 0.3588s ago] (1,)


...

2026-09-26 11:57:27,009 INFO sqlalchemy.engine.Engine [cached since 0.3627s ago] (3,)


2026-09-26 11:57:27,010 INFO sqlalchemy.engine.Engine ROLLBACK



exercise 2 - N+1 query count: 4
exercise 2 - selectinload query count: 2
exercise 3 - tracks left after deleting their playlist: []
Generating /tmp/alembic_exercise_wstahw3p/versions/23a76082f631_create_customers_table.py ...

  done
Generating /tmp/alembic_exercise_wstahw3p/versions/8f4cbcd97476_rename_name_to_full_name.py ...

  done
exercise 4 - customers columns after the rename migration: ['id', 'full_name']
exercise 4 - temp directory removed: True


* congratulation you finished this lesson

## what we learned
* many-to-many relationships: modelling the in-between association table as a plain `Table` (never its own mapped class), and wiring it up with `relationship(secondary=...)` on both sides
* the N+1 query problem: how lazy-loaded relationships can silently fire one extra query per object in a loop, how to actually OBSERVE it happening (an event listener counting statements, and `echo=True`), and how to fix it with `selectinload()` (a second bulk query) or `joinedload()` (a single JOIN query, remembering `.unique()`)
* cascades: that deleting a parent with no `cascade` configured tries to null out the children's foreign key by default (and fails loudly if that column is `NOT NULL`), versus `cascade="all, delete-orphan"`, which deletes the children automatically and safely
* joins and aggregates through the ORM: `select(Model).join(...)` for explicit joins, and `func.count()` + `.group_by()` + `.order_by()` for the same aggregate queries lesson 13 first showed us in raw SQL
* Alembic migrations: why `create_all()` alone is not enough for a real, evolving application, how a migration's `upgrade()`/`downgrade()` pair lets you ALTER an existing table without losing its data, and how a sequence of migrations builds up real schema evolution over time, tracked by alembic's own `alembic_version` table
* that this whole pattern (Django migrations, Rails' ActiveRecord, EF Migrations, Sequelize/Prisma/TypeORM, golang-migrate, Diesel, ...) and the N+1 problem (Hibernate's exact same issue) are close to universal across backend ecosystems, not python-specific quirks